# NIFTY Weekly 2-SD Short-Strangle Research

**Goal:** test whether selling NIFTY weekly options around an implied-volatility-derived 2-SD boundary has positive expectancy after costs.

This notebook is for research only. It does not assume that 2 SD implies a 95% real-world win probability.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT))
else:
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT))

from src.strangle_backtest import load_config, prepare_data, run_backtest, performance_report

CONFIG = ROOT / 'config.yaml'
DATA = ROOT / 'data' / 'sample_chain.csv'

In [ ]:
cfg, costs, raw = load_config(CONFIG)
data = prepare_data(DATA, cfg, slippage_points=costs.slippage_points_per_leg)
data.head()

## Baseline backtest

Wednesday 10:00 entry → nearest Tuesday expiry. Baseline uses pure 2-SD strike selection. The parameter-sweep stage will compare pure-SD, delta and hybrid selection.

In [ ]:
trades = run_backtest(data, cfg, costs)
report = performance_report(trades, cfg.starting_capital)
pd.Series(report)

In [ ]:
trades[['entry_timestamp','expiry','entry_spot','atm_iv','rv20','iv_rv_spread','put_strike','call_strike','initial_credit_rupees','exit_reason','net_pnl']]

In [ ]:
equity = cfg.starting_capital + trades['net_pnl'].cumsum()
equity.plot(title='Synthetic smoke-test equity curve')
plt.show()

## Replace the sample dataset

Point `DATA` at your historical NIFTY full-chain CSV/Parquet. Required columns: `timestamp, expiry, underlying, spot, strike, option_type`. Optional columns: `future, bid, ask, ltp, volume, oi, iv, delta`.

For serious validation, use quote-level bid/ask history rather than LTP-only data.

## Next research grid

Run a parameter sweep over SD = 1.5/1.75/2.0, target delta = 0.03/0.05/0.10, profit capture = 0.25/0.40/0.50/0.60, stop = 1.5/2.0/2.5/3.0, and entry weekday/time. Then use walk-forward validation to avoid selecting parameters on the same period used for evaluation.